# AgroPredict — Módulo 2: Integración de clima (NASA POWER)

Este notebook toma el maíz limpio del Módulo 1 y le **añade las variables climáticas** de cada
municipio y semestre, dejando la tabla lista para entrenar el modelo de rendimiento.

**Pasos:**
1. Obtener el **centroide (lat, lon)** de cada municipio (códigos DANE).
2. Consultar **NASA POWER** (API gratuita, sin clave) para temperatura, precipitación, radiación y humedad.
3. **Agregar el clima por semestre** (acumulados y promedios de la temporada A o B).
4. Hacer el **merge** con el maíz por `cod_mun + anio + semestre`.

> **Estructura de carpetas esperada:**
> ```
> proyecto/
>   data/      <- maiz_rendimiento_limpio.csv y centroides_municipios.csv aquí
>   notebook/  <- este notebook aquí
> ```
> Si tu estructura es otra, ajusta las rutas en la celda de configuración.

> ⏱️ **Importante:** el paso 2 descarga clima para ~1.063 municipios y tarda **20–40 min la primera vez**.
> Hay **caché en disco**, así que si se interrumpe puedes volver a correrlo y retoma donde quedó (y las
> siguientes corridas son instantáneas). Para una **prueba rápida**, pon `LIMITE_MUNICIPIOS = 20` abajo.

## 0. Configuración e importaciones

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import os

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# === RUTAS (relativas a la carpeta notebook/) ===
RUTA_MAIZ       = "../data/maiz_rendimiento_limpio.csv"
RUTA_CENTROIDES = "../data/centroides_municipios.csv"
DIR_CACHE       = "../data/cache_clima"      # clima diario por municipio (caché)
RUTA_SALIDA     = "../data/maiz_clima_modelo.csv"
os.makedirs(DIR_CACHE, exist_ok=True)

# === PARÁMETROS NASA POWER (comunidad AG = agroclimatología) ===
PARAMETROS = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'ALLSKY_SFC_SW_DWN', 'RH2M']
# T2M: temperatura media | T2M_MAX/MIN: máx/mín | PRECTOTCORR: precipitación (mm/día)
# ALLSKY_SFC_SW_DWN: radiación solar superficie | RH2M: humedad relativa (%)

PAUSA_API = 0.6   # segundos de cortesía entre llamadas a la API
REINTENTOS = 3    # reintentos si una llamada falla

# Para una PRUEBA rápida pon un número (p.ej. 20). Para TODOS los municipios, déjalo en None.
LIMITE_MUNICIPIOS = None

print("Configuración lista.")

## 1. Cargar maíz limpio y centroides de municipios

Cargamos el maíz del Módulo 1 (**leyendo los códigos como texto** para no perder los ceros iniciales)
y el archivo de centroides. Cada municipio tiene un punto (lat, lon) que usaremos para consultar el clima.

In [ ]:
# OJO: dtype=str en los códigos para conservar los ceros (05, 05051)
maiz = pd.read_csv(RUTA_MAIZ, dtype={'cod_dep': str, 'cod_mun': str})
maiz['anio'] = maiz['anio'].astype(int)

centroides = pd.read_csv(RUTA_CENTROIDES, dtype={'cod_dep': str, 'cod_mun': str})

print(f"Maíz: {len(maiz):,} filas | municipios con maíz: {maiz['cod_mun'].nunique()}")
print(f"Centroides disponibles: {len(centroides):,} municipios")

# Lista de municipios únicos a consultar, con su punto (lat, lon)
municipios = (maiz[['cod_mun', 'municipio']].drop_duplicates()
              .merge(centroides[['cod_mun', 'lat', 'lon']], on='cod_mun', how='left'))

sin_centroide = municipios['lat'].isna().sum()
print(f"Municipios de maíz a consultar: {len(municipios)}")
print(f"Municipios SIN centroide (se omitirán): {sin_centroide}")
municipios = municipios.dropna(subset=['lat', 'lon']).reset_index(drop=True)
municipios.head()

> **¿De dónde salen los centroides?** Se calcularon a partir de la capa municipal oficial del DANE
> (Marco Geoestadístico Nacional, MGN), tomando el centroide de cada polígono. La celda opcional al final
> del notebook muestra cómo regenerarlos desde un shapefile/GeoJSON si quieres rehacerlos tú mismo.

## 2. Consultar NASA POWER

Hacemos **una sola petición por municipio** cubriendo todo el periodo (2006–2018) y luego repartimos
los datos por semestre. NASA POWER es gratuita y no requiere clave de API.

Detalles de la implementación:
- **Caché en disco:** cada municipio se guarda en `data/cache_clima/`. Si ya está descargado, no se vuelve a pedir.
- **Valor faltante:** POWER usa `-999`; lo convertimos a `NaN`.
- **Reintentos** y pausa entre llamadas para ser amables con el servicio.

In [ ]:
def consultar_power(lat, lon, fecha_inicio, fecha_fin, parametros):
    # Una petición a NASA POWER (daily, punto). Devuelve un DataFrame diario.
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": ",".join(parametros),
        "community": "AG",
        "latitude": lat,
        "longitude": lon,
        "start": fecha_inicio,   # "AAAAMMDD"
        "end": fecha_fin,
        "format": "JSON",
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    registros = r.json()["properties"]["parameter"]
    df = pd.DataFrame(registros)
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    df.index.name = "fecha"
    return df.replace(-999, np.nan)


def obtener_clima_municipio(cod_mun, lat, lon, fecha_inicio, fecha_fin):
    # Devuelve el clima diario de un municipio, usando caché en disco.
    ruta = os.path.join(DIR_CACHE, f"clima_{cod_mun}.csv")
    if os.path.exists(ruta):
        return pd.read_csv(ruta, parse_dates=["fecha"], index_col="fecha")
    # No está en caché: descargar (con reintentos)
    for intento in range(1, REINTENTOS + 1):
        try:
            df = consultar_power(lat, lon, fecha_inicio, fecha_fin, PARAMETROS)
            df.to_csv(ruta)
            time.sleep(PAUSA_API)
            return df
        except Exception as e:
            if intento == REINTENTOS:
                raise
            time.sleep(2 * intento)  # backoff


# Rango de fechas según los años del maíz
INICIO = f"{maiz['anio'].min()}0101"
FIN    = f"{maiz['anio'].max()}1231"
print(f"Periodo a consultar: {INICIO} a {FIN}")

**Prueba de conexión:** descargamos un municipio para confirmar que la API responde.

In [ ]:
ejemplo = municipios.iloc[0]
prueba = obtener_clima_municipio(ejemplo['cod_mun'], ejemplo['lat'], ejemplo['lon'], INICIO, FIN)
print(f"Municipio de prueba: {ejemplo['municipio']} ({ejemplo['cod_mun']})")
print(f"Días descargados: {len(prueba)}  |  Columnas: {list(prueba.columns)}")
prueba.head()

## 3. Agregar el clima por semestre

Para el maíz (transitorio) interesa el clima de la **temporada de cultivo**, no del año entero:
- **Semestre A** = meses 1–6
- **Semestre B** = meses 7–12

Para cada municipio, año y semestre calculamos:
- `precip_total_mm`: lluvia **acumulada**
- `precip_dias_lluvia`: número de días con lluvia (> 1 mm)
- `temp_media_c`, `temp_max_media_c`, `temp_min_media_c`: temperaturas **promedio**
- `radiacion_media`, `humedad_media_pct`: radiación y humedad **promedio**

Agregamos *sobre la marcha* (municipio por municipio) para no cargar todo el clima diario en memoria.

In [ ]:
def agregar_por_semestre(daily, cod_mun):
    # Resume el clima diario de un municipio en filas (año, semestre).
    d = daily.copy()
    d['anio'] = d.index.year
    d['semestre'] = np.where(d.index.month <= 6, 'A', 'B')
    g = d.groupby(['anio', 'semestre'])
    out = pd.DataFrame({
        'precip_total_mm':    g['PRECTOTCORR'].sum(),
        'precip_dias_lluvia': g['PRECTOTCORR'].apply(lambda s: int((s > 1).sum())),
        'temp_media_c':       g['T2M'].mean(),
        'temp_max_media_c':   g['T2M_MAX'].mean(),
        'temp_min_media_c':   g['T2M_MIN'].mean(),
        'radiacion_media':    g['ALLSKY_SFC_SW_DWN'].mean(),
        'humedad_media_pct':  g['RH2M'].mean(),
    }).reset_index()
    out.insert(0, 'cod_mun', cod_mun)
    return out

In [ ]:
# Recorre todos los municipios: descarga (o lee de caché) y agrega por semestre.
objetivo = municipios if LIMITE_MUNICIPIOS is None else municipios.head(LIMITE_MUNICIPIOS)
print(f"Procesando {len(objetivo)} municipios...\n")

filas_clima, fallidos = [], []
for i, r in objetivo.iterrows():
    try:
        daily = obtener_clima_municipio(r['cod_mun'], r['lat'], r['lon'], INICIO, FIN)
        filas_clima.append(agregar_por_semestre(daily, r['cod_mun']))
    except Exception as e:
        fallidos.append((r['cod_mun'], str(e)[:80]))
    if (i + 1) % 50 == 0 or (i + 1) == len(objetivo):
        print(f"  {i+1}/{len(objetivo)} municipios procesados...")

clima_semestral = pd.concat(filas_clima, ignore_index=True)
clima_semestral['anio'] = clima_semestral['anio'].astype(int)

print(f"\nClima semestral: {clima_semestral.shape[0]:,} filas (municipio × año × semestre)")
print(f"Municipios fallidos: {len(fallidos)}")
if fallidos:
    print("  (puedes re-ejecutar esta celda; el caché evita repetir lo ya descargado)")
clima_semestral.head()

## 4. Merge con el maíz

Unimos el clima al maíz por la llave `cod_mun + anio + semestre`. Cada registro de maíz queda
acompañado del clima de su municipio y temporada exactos.

In [ ]:
tabla_modelo = maiz.merge(clima_semestral, on=['cod_mun', 'anio', 'semestre'], how='left')

con_clima = tabla_modelo['precip_total_mm'].notna().sum()
print(f"Tabla de modelado: {len(tabla_modelo):,} filas")
print(f"Filas con clima pegado: {con_clima:,} ({100*con_clima/len(tabla_modelo):.1f}%)")
print(f"Filas sin clima: {len(tabla_modelo)-con_clima:,}")
print("\nColumnas climáticas añadidas:")
print([c for c in tabla_modelo.columns if c not in maiz.columns])
tabla_modelo.head()

## 5. Vistazo a la relación clima–rendimiento

Antes de modelar, miramos si el clima muestra relación con el rendimiento.

In [ ]:
datos = tabla_modelo.dropna(subset=['precip_total_mm', 'rendimiento_tha'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(datos['precip_total_mm'], datos['rendimiento_tha'], s=6, alpha=0.2)
axes[0].set_xlabel('Precipitación acumulada del semestre (mm)')
axes[0].set_ylabel('Rendimiento (t/ha)')
axes[0].set_title('Precipitación vs rendimiento')

axes[1].scatter(datos['temp_media_c'], datos['rendimiento_tha'], s=6, alpha=0.2, color='#DD8452')
axes[1].set_xlabel('Temperatura media del semestre (°C)')
axes[1].set_ylabel('Rendimiento (t/ha)')
axes[1].set_title('Temperatura vs rendimiento')
plt.tight_layout(); plt.show()

In [ ]:
# Correlación de cada variable climática con el rendimiento
clim_cols = ['precip_total_mm','precip_dias_lluvia','temp_media_c',
             'temp_max_media_c','temp_min_media_c','radiacion_media','humedad_media_pct']
corr = datos[clim_cols + ['rendimiento_tha']].corr()['rendimiento_tha'].drop('rendimiento_tha')
print("Correlación con el rendimiento (Pearson):")
print(corr.sort_values(ascending=False).round(3).to_string())

## 6. Exportar la tabla de modelado

In [ ]:
tabla_modelo.to_csv(RUTA_SALIDA, index=False, encoding='utf-8')
print(f"Guardado: {RUTA_SALIDA}  ({len(tabla_modelo):,} filas, {tabla_modelo.shape[1]} columnas)")
print("\nRecuerda releerlo con dtype={'cod_dep': str, 'cod_mun': str} para conservar los ceros.")

## Siguiente paso: entrenar el modelo

Con `maiz_clima_modelo.csv` ya tienes la tabla con variable objetivo (`rendimiento_tha`) y predictoras
(clima + sistema productivo + ubicación + año/semestre). El siguiente módulo sería:

1. Codificar las variables categóricas (`departamento`, `desagregacion`, `semestre`).
2. Separar entrenamiento / prueba (idealmente **por años**, p. ej. entrenar 2006–2016 y validar 2017–2018).
3. Entrenar un modelo de regresión (Random Forest o Gradient Boosting suelen funcionar muy bien aquí).
4. Evaluar con RMSE / MAE y revisar la importancia de variables.

Y en paralelo, el **módulo de precios (SIPSA)** para convertir el rendimiento estimado en estimación de ganancia.

---
## Apéndice (opcional): regenerar los centroides desde un shapefile DANE

**No necesitas correr esto** si ya tienes `centroides_municipios.csv`. Se incluye para que sepas
cómo se generó y puedas rehacerlo con la capa oficial del DANE (Marco Geoestadístico Nacional).

In [ ]:
# OPCIONAL — requiere shapely. Lee un GeoJSON/shapefile municipal y calcula centroides.
#
# import json
# from shapely.geometry import shape
#
# d = json.load(open("../data/co_2018_MGN_MPIO_POLITICO.geojson"))
# filas = []
# for f in d["features"]:
#     p = f["properties"]
#     c = shape(f["geometry"]).centroid
#     filas.append({
#         "cod_mun": p["MPIO_CCNCT"],          # código DANE de 5 dígitos
#         "municipio": p["MPIO_CNMBR"],
#         "cod_dep": p["DPTO_CCDGO"],
#         "departamento": p["DPTO_CNMBR"],
#         "lat": round(c.y, 5), "lon": round(c.x, 5),
#     })
# pd.DataFrame(filas).to_csv("../data/centroides_municipios.csv", index=False, encoding="utf-8")
#
# Para un shapefile (.shp) directamente, usa geopandas:
#   import geopandas as gpd
#   gdf = gpd.read_file("ruta/al/MGN_MPIO_POLITICO.shp")
#   gdf["lat"] = gdf.geometry.centroid.y
#   gdf["lon"] = gdf.geometry.centroid.x
print("Celda de referencia (comentada).")